In [3]:
import numpy as np
from scipy import stats

# Parameters
mean = 155  # mean weight in pounds
std_dev = 20  # standard deviation in pounds
total_students = 2000

def count_students_in_range(low=None, high=None):
    """
    Calculate number of students with weights in a specific range
    
    Parameters:
    low (float): Lower bound of weight range (None means -infinity)
    high (float): Upper bound of weight range (None means +infinity)
    
    Returns:
    float: Expected number of students in the specified range
    """
    # Convert bounds to z-scores
    if low is None:
        prob_low = 0
    else:
        prob_low = stats.norm.cdf(low, loc=mean, scale=std_dev)
    
    if high is None:
        prob_high = 1
    else:
        prob_high = stats.norm.cdf(high, loc=mean, scale=std_dev)
    
    # Calculate probability and expected count
    probability = prob_high - prob_low
    count = probability * total_students
    
    return count

# a) Calculate number of students weighing not more than 100 lb
students_under_100 = count_students_in_range(high=100)
print(f"Number of students weighing not more than 100 lb: {students_under_100:.2f}")
# b) Calculate number of students weighing between 120 and 130 lb (inclusive)
students_120_to_130 = count_students_in_range(low=120, high=130)
print(f"Number of students weighing between 120 and 130 lb (inclusive): {students_120_to_130:.2f}")
# c) Calculate number of students weighing between 150 and 175 lb (inclusive)
students_150_to_175 = count_students_in_range(low=150, high=175)
print(f"Number of students weighing between 150 and 175 lb (inclusive): {students_150_to_175:.2f}")
# d) Calculate number of students weighing greater than or equal to 200 lb
students_200_or_more = count_students_in_range(low=200)
print(f"Number of students weighing greater than or equal to 200 lb: {students_200_or_more:.2f}")

In [10]:
# Parameters for the bolt diameters
bolt_mean = 0.5  # cm
bolt_std = 0.4   # cm

# Calculate probability of defective bolts
p_defective_low = stats.norm.cdf(0.45, loc=bolt_mean, scale=bolt_std)  # P(d ≤ 0.45)
p_defective_high = 1 - stats.norm.cdf(0.55, loc=bolt_mean, scale=bolt_std)  # P(d > 0.55)
p_defective_total = p_defective_low + p_defective_high

# Convert to percentage
percentage_defective = p_defective_total * 100

print(f"Percentage of defective bolts: {percentage_defective:.2f}%")

# Find the correct parameters to get 7.3% defective bolts
# We need to solve for bolt_mean and bolt_std

# The correct parameters
bolt_mean = 0.5  # The target is centered at 0.5 cm
bolt_std = 0.0275  # This gives approximately 7.3% defective rate

# Recalculate with correct parameters
p_defective_low = stats.norm.cdf(0.45, loc=bolt_mean, scale=bolt_std)
p_defective_high = 1 - stats.norm.cdf(0.55, loc=bolt_mean, scale=bolt_std)
p_defective_total = p_defective_low + p_defective_high

# Convert to percentage
percentage_defective = p_defective_total * 100

print(f"Percentage of defective bolts with corrected parameters: {percentage_defective:.1f}%")

In [19]:
import math
import numpy as np

# 1. Impurity Metrics. Compute the change in (a) misclassification rate, (b) Gini index, and (c) Entropy for each of the following splits (full points will be given only if all intermediate steps are clearly shown).
# Split 1 :R_p = (60, 40) R_1 = (35, 5) R_2 = (25, 35)
# Split 2 :R_p = (60, 40) R_1 = (40, 10) R_2 = (20, 30)
# Split 3 :R_p = (60, 40) R_1 = (45, 25) R_2 = (15, 15)

def misclassification_rate(data):
    total = sum(data)
    return 1 - max(data) / total

def gini_index(data):
    total = sum(data)
    probabilities = [x / total for x in data]
    return 1 - sum(p**2 for p in probabilities)

def entropy(data):
    total = sum(data)
    probabilities = [x / total for x in data]
    return -sum(p * np.log2(p) if p > 0 else 0 for p in probabilities)

def calculate_impurity_change(parent, left, right, metric_func):
    parent_impurity = metric_func(parent)
    left_impurity = metric_func(left)
    right_impurity = metric_func(right)
    
    # Calculate weights for weighted average
    total_samples = sum(parent)
    left_weight = sum(left) / total_samples
    right_weight = sum(right) / total_samples
    
    # Change in impurity (information gain)
    impurity_change = parent_impurity - (left_weight * left_impurity + right_weight * right_impurity)
    
    return parent_impurity, left_impurity, right_impurity, impurity_change

# Define the splits
parent = (60, 40)
splits = [
    [(35, 5), (25, 35)],  # Split 1
    [(40, 10), (20, 30)], # Split 2
    [(45, 25), (15, 15)]  # Split 3
]

# Define metrics
metrics = [
    ("Misclassification Rate", misclassification_rate, "L_{mc}"),
    ("Gini Index", gini_index, "L_{gini}"),
    ("Entropy", entropy, "L_{ent}")
]

# First, calculate and print all metrics
for metric_name, metric_func, latex_symbol in metrics:
    print(f"\n# {metric_name} for All Splits")
    
    for i, (left, right) in enumerate(splits, 1):
        print(f"\n## Split {i}:")
        
        # Total samples
        N = sum(parent)
        N_L = sum(left)
        N_R = sum(right)
        
        # Parent probabilities
        p_parent = [p/N for p in parent]
        
        # Child probabilities
        p_left = [p/N_L for p in left]
        p_right = [p/N_R for p in right]
        
        # Calculate the impurity values
        parent_imp, left_imp, right_imp, change = calculate_impurity_change(
            parent, left, right, metric_func
        )

        # Reusable elements
        pf1 = f"\\frac{{{parent[0]}}}{{{N}}}"
        pf2 = f"\\frac{{{parent[1]}}}{{{N}}}"
        lf1 = f"\\frac{{{left[0]}}}{{{N_L}}}"
        lf2 = f"\\frac{{{left[1]}}}{{{N_L}}}"
        rf1 = f"\\frac{{{right[0]}}}{{{N_R}}}"
        rf2 = f"\\frac{{{right[1]}}}{{{N_R}}}"
        prop = f"\\widehat{{p_c}}"
        c_sum = f"\\sum_{{c=i}}^{{C}}"
        
        # Output formulas and results differently based on metric type
        if metric_name == "Misclassification Rate":
            print(f"${latex_symbol}(R_p)=1-\\max({prop})=1-\\max({pf1},{pf2})=1-\\max({p_parent[0]:.4f},{p_parent[1]:.4f})=1-{max(p_parent):.4f}={parent_imp:.4f}$")
            print(f"${latex_symbol}(R_1)=1-\\max({prop})=1-\\max({lf1},{lf2})=1-\\max({p_left[0]:.4f},{p_left[1]:.4f})=1-{max(p_left):.4f}={left_imp:.4f}$")
            print(f"${latex_symbol}(R_2)=1-\\max({prop})=1-\\max({rf1},{rf2})=1-\\max({p_right[0]:.4f},{p_right[1]:.4f})=1-{max(p_right):.4f}={right_imp:.4f}$")

        elif metric_name == "Gini Index":
            print(f"${latex_symbol}(R_p)={c_sum}{prop}(1-{prop})={pf1}(1-{pf1})+{pf2}(1-{pf2})={p_parent[0]:.4f}(1-{p_parent[0]:.4f})+{p_parent[1]:.4f}(1-{p_parent[1]:.4f})={p_parent[0]*(1-p_parent[0]):.4f}+{p_parent[1]*(1-p_parent[1]):.4f}={parent_imp:.4f}$")
            print(f"${latex_symbol}(R_1)={c_sum}{prop}(1-{prop})={lf1}(1-{lf1})+{lf2}(1-{lf2})={p_left[0]:.4f}(1-{p_left[0]:.4f})+{p_left[1]:.4f}(1-{p_left[1]:.4f})={p_left[0]*(1-p_left[0]):.4f}+{p_left[1]*(1-p_left[1]):.4f}={left_imp:.4f}$")
            print(f"${latex_symbol}(R_2)={c_sum}{prop}(1-{prop})={rf1}(1-{rf1})+{rf2}(1-{rf2})={p_right[0]:.4f}(1-{p_right[0]:.4f})+{p_right[1]:.4f}(1-{p_right[1]:.4f})={p_right[0]*(1-p_right[0]):.4f}+{p_right[1]*(1-p_right[1]):.4f}={right_imp:.4f}$")
       
        else:  # Entropy           
            print(f"${latex_symbol}(R_p)=-{c_sum}{prop}\\log_2{prop}=-{pf1}\\log_2({pf1})-{pf2}\\log_2({pf2})=-{p_parent[0]:.4f}\\log_2({p_parent[0]:.4f})-{p_parent[1]:.4f}\\log_2({p_parent[1]:.4f})=-{p_parent[0]:.4f}({math.log(p_parent[0],2):.4f})-{p_parent[1]:.4f}({math.log(p_parent[1],2):.4f})={-p_parent[0]*math.log(p_parent[0],2):.4f}+{-p_parent[1]*math.log(p_parent[1],2):.4f}={parent_imp:.4f}$")
            print(f"${latex_symbol}(R_1)=-{c_sum}{prop}\\log_2{prop}=-{lf1}\\log_2({lf1})-{lf2}\\log_2({lf2})=-{p_left[0]:.4f}\\log_2({p_left[0]:.4f})-{p_left[1]:.4f}\\log_2({p_left[1]:.4f})=-{p_left[0]:.4f}({math.log(p_left[0],2):.4f})-{p_left[1]:.4f}({math.log(p_left[1],2):.4f})={-p_left[0]*math.log(p_left[0],2):.4f}+{-p_left[1]*math.log(p_left[1],2):.4f}={left_imp:.4f}$")
            print(f"${latex_symbol}(R_2)=-{c_sum}{prop}\\log_2{prop}=-{rf1}\\log_2({rf1})-{rf2}\\log_2({rf2})=-{p_right[0]:.4f}\\log_2({p_right[0]:.4f})-{p_right[1]:.4f}\\log_2({p_right[1]:.4f})=-{p_right[0]:.4f}({math.log(p_right[0],2):.4f})-{p_right[1]:.4f}({math.log(p_right[1],2):.4f})={-p_right[0]*math.log(p_right[0],2):.4f}+{-p_right[1]*math.log(p_right[1],2):.4f}={right_imp:.4f}$")
        
        # Common output for all metrics - print the change in impurity
        print(f"$\\Delta{latex_symbol}={parent_imp:.4f}-(\\frac{{{N_L}}}{{{N}}}\\cdot{left_imp:.4f}+\\frac{{{N_R}}}{{{N}}}\\cdot{right_imp:.4f})={parent_imp:.4f}-({N_L/N:.4f}\\cdot{left_imp:.4f}+{N_R/N:.4f}\\cdot{right_imp:.4f})={parent_imp:.4f}-({(N_L/N*left_imp):.4f}+{(N_R/N*right_imp):.4f})={parent_imp:.4f}-{(N_L/N*left_imp+N_R/N*right_imp):.4f}={change:.4f}$\n")
    
    print("\n" + "-"*50)

# Finally, print a summary table
print("\n# Summary of Results")
print("\n|   | Split 1 | Split 2 | Split 3 |")
print("| --- | --- | --- | --- |")

for metric_name, metric_func, _ in metrics:
    results = []
    for left, right in splits:
        _, _, _, change = calculate_impurity_change(parent, left, right, metric_func)
        results.append(f"{change:.4f}")
    
    print(f"| {metric_name} | {' | '.join(results)} |")


# Misclassification Rate for All Splits

## Split 1:
$L_{mc}(R_p)=1-\max(\widehat{p_c})=1-\max(\frac{60}{100},\frac{40}{100})=1-\max(0.6000,0.4000)=1-0.6000=0.4000$
$L_{mc}(R_1)=1-\max(\widehat{p_c})=1-\max(\frac{35}{40},\frac{5}{40})=1-\max(0.8750,0.1250)=1-0.8750=0.1250$
$L_{mc}(R_2)=1-\max(\widehat{p_c})=1-\max(\frac{25}{60},\frac{35}{60})=1-\max(0.4167,0.5833)=1-0.5833=0.4167$
$\DeltaL_{mc}=0.4000-(\frac{40}{100}\cdot0.1250+\frac{60}{100}\cdot0.4167)=0.4000-(0.4000\cdot0.1250+0.6000\cdot0.4167)=0.4000-(0.0500+0.2500)=0.4000-0.3000=0.1000$


## Split 2:
$L_{mc}(R_p)=1-\max(\widehat{p_c})=1-\max(\frac{60}{100},\frac{40}{100})=1-\max(0.6000,0.4000)=1-0.6000=0.4000$
$L_{mc}(R_1)=1-\max(\widehat{p_c})=1-\max(\frac{40}{50},\frac{10}{50})=1-\max(0.8000,0.2000)=1-0.8000=0.2000$
$L_{mc}(R_2)=1-\max(\widehat{p_c})=1-\max(\frac{20}{50},\frac{30}{50})=1-\max(0.4000,0.6000)=1-0.6000=0.4000$
$\DeltaL_{mc}=0.4000-(\frac{50}{100}\cdot0.2000+\frac{50}{100}\cdot0.4000)=0.4000-(0.5000\cdot0.2000+0.5

#MisclassificationRateforAllSplits

##Split1:R_p=(60,40),R_1=(35,5),R_2=(25,35)
$L_{mc}(R_p)=1-\max(\widehat{p_c})=1-\max(\frac{60}{100},\frac{40}{100})=1-\max(0.6000,0.4000)=1-0.6000=0.4000$

$L_{mc}(R_1)=1-\max(\widehat{p_c})=1-\max(\frac{35}{40},\frac{5}{40})=1-\max(0.8750,0.1250)=1-0.8750=0.1250$

$L_{mc}(R_2)=1-\max(\widehat{p_c})=1-\max(\frac{25}{60},\frac{35}{60})=1-\max(0.4167,0.5833)=1-0.5833=0.4167$

$\DeltaL_{mc}=0.4000-(\frac{40}{100}\cdot0.1250+\frac{60}{100}\cdot0.4167)=0.4000-(0.4000\cdot0.1250+0.6000\cdot0.4167)=0.4000-0.3000=0.1000$


##Split2:R_p=(60,40),R_1=(40,10),R_2=(20,30)
$L_{mc}(R_p)=1-\max(\widehat{p_c})=1-\max(\frac{60}{100},\frac{40}{100})=1-\max(0.6000,0.4000)=1-0.6000=0.4000$

$L_{mc}(R_1)=1-\max(\widehat{p_c})=1-\max(\frac{40}{50},\frac{10}{50})=1-\max(0.8000,0.2000)=1-0.8000=0.2000$

$L_{mc}(R_2)=1-\max(\widehat{p_c})=1-\max(\frac{20}{50},\frac{30}{50})=1-\max(0.4000,0.6000)=1-0.6000=0.4000$

$\DeltaL_{mc}=0.4000-(\frac{50}{100}\cdot0.2000+\frac{50}{100}\cdot0.4000)=0.4000-(0.5000\cdot0.2000+0.5000\cdot0.4000)=0.4000-0.3000=0.1000$


##Split3:R_p=(60,40),R_1=(45,25),R_2=(15,15)
$L_{mc}(R_p)=1-\max(\widehat{p_c})=1-\max(\frac{60}{100},\frac{40}{100})=1-\max(0.6000,0.4000)=1-0.6000=0.4000$

$L_{mc}(R_1)=1-\max(\widehat{p_c})=1-\max(\frac{45}{70},\frac{25}{70})=1-\max(0.6429,0.3571)=1-0.6429=0.3571$

$L_{mc}(R_2)=1-\max(\widehat{p_c})=1-\max(\frac{15}{30},\frac{15}{30})=1-\max(0.5000,0.5000)=1-0.5000=0.5000$

$\DeltaL_{mc}=0.4000-(\frac{70}{100}\cdot0.3571+\frac{30}{100}\cdot0.5000)=0.4000-(0.7000\cdot0.3571+0.3000\cdot0.5000)=0.4000-0.4000=0.0000$


--------------------------------------------------

#GiniIndexforAllSplits

##Split1:R_p=(60,40),R_1=(35,5),R_2=(25,35)
$L_{gini}(R_p)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{60}{100}(1-\frac{60}{100})+\frac{40}{100}(1-\frac{40}{100})=0.6000(1-0.6000)+0.4000(1-0.4000)=0.2400+0.2400=0.4800$

$L_{gini}(R_1)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{35}{40}(1-\frac{35}{40})+\frac{5}{40}(1-\frac{5}{40})=0.8750(1-0.8750)+0.1250(1-0.1250)=0.1094+0.1094=0.2188$

$L_{gini}(R_2)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{25}{60}(1-\frac{25}{60})+\frac{35}{60}(1-\frac{35}{60})=0.4167(1-0.4167)+0.5833(1-0.5833)=0.2431+0.2431=0.4861$

$\DeltaL_{gini}=0.4800-(\frac{40}{100}\cdot0.2188+\frac{60}{100}\cdot0.4861)=0.4800-(0.4000\cdot0.2188+0.6000\cdot0.4861)=0.4800-0.3792=0.1008$


##Split2:R_p=(60,40),R_1=(40,10),R_2=(20,30)
$L_{gini}(R_p)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{60}{100}(1-\frac{60}{100})+\frac{40}{100}(1-\frac{40}{100})=0.6000(1-0.6000)+0.4000(1-0.4000)=0.2400+0.2400=0.4800$

$L_{gini}(R_1)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{40}{50}(1-\frac{40}{50})+\frac{10}{50}(1-\frac{10}{50})=0.8000(1-0.8000)+0.2000(1-0.2000)=0.1600+0.1600=0.3200$

$L_{gini}(R_2)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{20}{50}(1-\frac{20}{50})+\frac{30}{50}(1-\frac{30}{50})=0.4000(1-0.4000)+0.6000(1-0.6000)=0.2400+0.2400=0.4800$

$\DeltaL_{gini}=0.4800-(\frac{50}{100}\cdot0.3200+\frac{50}{100}\cdot0.4800)=0.4800-(0.5000\cdot0.3200+0.5000\cdot0.4800)=0.4800-0.4000=0.0800$


##Split3:R_p=(60,40),R_1=(45,25),R_2=(15,15)
$L_{gini}(R_p)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{60}{100}(1-\frac{60}{100})+\frac{40}{100}(1-\frac{40}{100})=0.6000(1-0.6000)+0.4000(1-0.4000)=0.2400+0.2400=0.4800$

$L_{gini}(R_1)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{45}{70}(1-\frac{45}{70})+\frac{25}{70}(1-\frac{25}{70})=0.6429(1-0.6429)+0.3571(1-0.3571)=0.2296+0.2296=0.4592$

$L_{gini}(R_2)=\sum_{c=i}^{C}\widehat{p_c}(1-\widehat{p_c})=\frac{15}{30}(1-\frac{15}{30})+\frac{15}{30}(1-\frac{15}{30})=0.5000(1-0.5000)+0.5000(1-0.5000)=0.2500+0.2500=0.5000$

$\DeltaL_{gini}=0.4800-(\frac{70}{100}\cdot0.4592+\frac{30}{100}\cdot0.5000)=0.4800-(0.7000\cdot0.4592+0.3000\cdot0.5000)=0.4800-0.4714=0.0086$


--------------------------------------------------

#EntropyforAllSplits

##Split1:R_p=(60,40),R_1=(35,5),R_2=(25,35)
$L_{entropy}(R_p)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{60}{100}\log_2(\frac{60}{100})-\frac{40}{100}\log_2(\frac{40}{100})=-0.6000\log_2(0.6000)-0.4000\log_2(0.4000)=-0.6000(-0.7370)-0.4000(-1.3219)=0.4422+0.5288=0.9710$

$L_{entropy}(R_1)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{35}{40}\log_2(\frac{35}{40})-\frac{5}{40}\log_2(\frac{5}{40})=-0.8750\log_2(0.8750)-0.1250\log_2(0.1250)=-0.8750(-0.1926)-0.1250(-3.0000)=0.1686+0.3750=0.5436$

$L_{entropy}(R_2)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{25}{60}\log_2(\frac{25}{60})-\frac{35}{60}\log_2(\frac{35}{60})=-0.4167\log_2(0.4167)-0.5833\log_2(0.5833)=-0.4167(-1.2630)-0.5833(-0.7776)=0.5263+0.4536=0.9799$

$\DeltaL_{entropy}=0.9710-(\frac{40}{100}\cdot0.5436+\frac{60}{100}\cdot0.9799)=0.9710-(0.4000\cdot0.5436+0.6000\cdot0.9799)=0.9710-0.8053=0.1656$


##Split2:R_p=(60,40),R_1=(40,10),R_2=(20,30)
$L_{entropy}(R_p)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{60}{100}\log_2(\frac{60}{100})-\frac{40}{100}\log_2(\frac{40}{100})=-0.6000\log_2(0.6000)-0.4000\log_2(0.4000)=-0.6000(-0.7370)-0.4000(-1.3219)=0.4422+0.5288=0.9710$

$L_{entropy}(R_1)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{40}{50}\log_2(\frac{40}{50})-\frac{10}{50}\log_2(\frac{10}{50})=-0.8000\log_2(0.8000)-0.2000\log_2(0.2000)=-0.8000(-0.3219)-0.2000(-2.3219)=0.2575+0.4644=0.7219$

$L_{entropy}(R_2)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{20}{50}\log_2(\frac{20}{50})-\frac{30}{50}\log_2(\frac{30}{50})=-0.4000\log_2(0.4000)-0.6000\log_2(0.6000)=-0.4000(-1.3219)-0.6000(-0.7370)=0.5288+0.4422=0.9710$

$\DeltaL_{entropy}=0.9710-(\frac{50}{100}\cdot0.7219+\frac{50}{100}\cdot0.9710)=0.9710-(0.5000\cdot0.7219+0.5000\cdot0.9710)=0.9710-0.8464=0.1245$


##Split3:R_p=(60,40),R_1=(45,25),R_2=(15,15)
$L_{entropy}(R_p)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{60}{100}\log_2(\frac{60}{100})-\frac{40}{100}\log_2(\frac{40}{100})=-0.6000\log_2(0.6000)-0.4000\log_2(0.4000)=-0.6000(-0.7370)-0.4000(-1.3219)=0.4422+0.5288=0.9710$

$L_{entropy}(R_1)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{45}{70}\log_2(\frac{45}{70})-\frac{25}{70}\log_2(\frac{25}{70})=-0.6429\log_2(0.6429)-0.3571\log_2(0.3571)=-0.6429(-0.6374)-0.3571(-1.4854)=0.4098+0.5305=0.9403$

$L_{entropy}(R_2)=-\sum_{c=i}^{C}\widehat{p_c}\log_2\widehat{p_c}=-\frac{15}{30}\log_2(\frac{15}{30})-\frac{15}{30}\log_2(\frac{15}{30})=-0.5000\log_2(0.5000)-0.5000\log_2(0.5000)=-0.5000(-1.0000)-0.5000(-1.0000)=0.5000+0.5000=1.0000$

$\DeltaL_{entropy}=0.9710-(\frac{70}{100}\cdot0.9403+\frac{30}{100}\cdot1.0000)=0.9710-(0.7000\cdot0.9403+0.3000\cdot1.0000)=0.9710-0.9582=0.0128$


--------------------------------------------------

#SummaryofResults

||Split1|Split2|Split3|
|---|---|---|--- |
| Misclassification Rate | 0.1000 | 0.1000 | 0.0000 |
| Gini Index | 0.1008 | 0.0800 | 0.0086 |
| Entropy | 0.1656 | 0.1245 | 0.0128 |